#### **Step 3 - Dense Retrieval (Semantic Search)**

- **The code idea:** Dense retrieval catches similarities between a query and a chunk that use completely different words 

- Unlike keyword search which looks for exact word matches, semantic search relies on semantic matches between a chunk and its query calculated through cosine similarity 

- **Steps**
    1. **Embed every chunk into a vector using a sentence emebdding model**
        - Each chunk's full text is passed through `all-MiniLM-L6-v2` (embedding model) and converted into a single 384 dimensional vector - one vector per chunk

        - This model is trained with a contrastive objective: pairs of sentences with similar meaning are pushed to have vectors pointing in similar directions, dissimilar pairs pushed apart 

        - Note: Sentence-transformers embedding models are purpose-built for this task - using GPT-2's embedding model will perform poorly since it's specifically made to predict the next word rather than being optimised to make similar meanings produce similar vectors
 
    2. **Embed the query into a vector using the same model** 
        - At search time, the question (query) is passed through the exact same embedding model, producing its own vector

        - Using the same model is essential - it places the query vector into the same vector space as all the chunk vectors, making direct comparison meaningful 
    
    3. **Normalise all vectors to unit length**
        - Every embedding (chunks and query) is rescaled so its magnitude equals exactyl 1, while preserving its direction 

        - This steps is what allows the cosine similarity to be mathematically equivalent to the dot product because we simply divide by 1 instead of having to compute the magnitudes of both the chunks and query

    4. **Compute the cosine similarity between the query vector and every chunk vector (dot product rearranged)**
        - Cosine similarity (rearranged formula for the dot product) measures the angle between two vectors, ignoring magnitude

            $\cos(\theta) = \frac{A \cdot B}{\lVert A \rVert \lVert B \rVert} = A \cdot B$ 
            
            - As magnitudes are normalised to 1 we can calculate the cosine similarity using a plain, cheap dot product

            - This cosine similarity score is what tells us how closely related a given chunk is to the query ranging from -1 to 1 (opposite meaning to identical meaning)

    5. **Rank chunks by similarity score, highest first**
        - Each chunk now has a sintgle similarity score against the query

        - Chunks are sorted so the most semantically similar chunk appears first 
        
    6. **Return the top-k chunks**
        - Only the top-k highest-scoring chunks are kept and returned, in the same `(chunk_dict}, score)` tuple format as `SparseIndex.search()` - this consistent output is what allows RRF fusion (the next step) to combine results from both BM25 and semantic search

In [1]:
# used to embed the chunks into vectors 
from sentence_transformers import SentenceTransformer 

# used to perform cosine similarity comptuations between the 2 vectors 
import faiss 

class DenseIndex:
    def __init__(self, chunks, model_name="all-MiniLM-L6-v2"):
        # intialise the chunks that we will use throughout 
        # extract the text for each chunk to create a list of chunk strings - this is what we will pass to be embedded
        self.chunks = chunks 
        text_chunks = [chunk['text'] for chunk in chunks]

        # intialise the embedding model we will be using 
        self.model = SentenceTransformer(model_name)

        # embed the text chunks into vectors - a vector for each chunk
        # returns a list of vectors 384 dimensional vectors for each chunk
        embedded_chunks = self.model.encode(text_chunks)

        # takes the embedded chunks and normalises them to a magnitude of 1
        # this way, the cosine similarty equals the dot product since the matgnitude of the vectors are simply 1 
        faiss.normalize_L2(embedded_chunks) # normalises in place
        
        # create a data structure to store all the chunk vectors inside - pass in the size of the vectors to create an index that can store vectors of the required size
        # this is useful as later given a new query vector, it can quickly compare against all of them to determine the most similar vector 
        # store it on self since we will be using this index of embedded chunks to compute cosine similarity in search()
        self.index = faiss.IndexFlatIP(embedded_chunks.shape[1]) 

        # populate the index with the normalised vectors 
        self.index.add(embedded_chunks)

    def search(self, query, k=5):
        # embed the query using the same embedding model as the chunks
        # we pass the query as a list of a single string so the dimensions match the 2d array of chunks
        embedded_query = self.model.encode([query])

        # normalise the query 
        faiss.normalize_L2(embedded_query)

        # now perform a search on the index of chunks against the query
        # this will compute the cosine similarity between each chunk and the query
        # returns the top k most similar chunks to the query: their scores and their positions (indices) in self.chunks
        scores, indices = self.index.search(embedded_query, k) 
        # this is exactly why we store the embeded chunks in an index - so we can search it with respect to a query

        # to store the (chunk_dict, score) tuples
        result = []

        # populate the result list
        # indices is a list of the corresponding indices of the chunk
        # scores is a list of the scores for the correspondding chunks with the index denoted by the same index in indices
        # zip lets us iterate over these 2 lists simultaneously accessing the score of the chunk and the index of the chunk to retrieve the dict and pair it with the score
        for score, chunk_idx in zip(scores[0], indices[0]):
            result.append((self.chunks[chunk_idx], float(score)))
        return result

**`DenseIndex` Intuitive Summary**

**The Big Picture:** For every chunk in the list of chunks we want to embed them into vectors, and then embed the query into a vector so that we can compute the cosine similarity between all chunks and the query vector. Using these cosine similarity scores, we will return the chunks with the highest sscore - most semantically similar to the query

- **Build time (`__init__`)**
    - We want to obtain a list of our chunk strings and then embed them into vectors using the embedding model

    - Normalise the embedded vectors to have a magnitude of 1 - this simplifies computation of cosine similarity to simply a dot product

    - Create an `index` Data Structure that allows us to store these embedded chunks and efficiently compute cosine similarity calculations during `search()` against the given query vector - populate the `index` with the normalised, embedded chunks

- **Query time (`.search()`)** 
    - We want to then embed the query as well into a vector using the exact same embedding model (pass query as a list of a single string so dimensions match the chunks we embedded)

    - Normalise the embedded query vector 

    - Compute the cosine similarities against the query vector using `.search()` on the `index` which will return a list of scores and a list of chunk indices 

    - Return a list of tuples pairing the `(chunk_dict, score)` exactly like in `sparse_index()`

**Testing**

In [2]:
from chunking import load_documents, chunk_documents 
docs = load_documents("../data")
chunks = chunk_documents(docs)

dense = DenseIndex(chunks)
dense.search('How many world championships has Faker won?', k=2)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[({'chunk_id': 'faker.txt_chunk2',
   'doc_id': 'faker.txt',
   'text': ' reinventing his role. He shifted from a pure carry into a strategic leader who set the tempo of the whole team, and he kept adapting to new champions and new patches every season.\n\nIn 2023 he lifted his fourth World Championship trophy, with T1 beating Weibo Gaming in Seoul, seven years after his previous one. In 2024 the team won again, defeating Bilibili Gaming in London in a close five-game series, and in 2025 T1 took a third consecutive title against KT Rolster. That gives Faker a record six World Championship titles, more than any other player in the history of the game.\n\nOutside of Worlds, Faker has won the Mid-Season Invitational twice, in 2016 and 2017'},
  0.7315964698791504),
 ({'chunk_id': 'faker.txt_chunk1',
   'doc_id': 'faker.txt',
   'text': ' Club in the final held in Los Angeles. He was only seventeen years old. He won his second World Championship in 2015 against KOO Tigers, and his third in